# PyDI Data Integration Workflow: Companies

This notebook demonstrates comprehensive data integration using PyDI. We'll work with companies datasets to showcase the data integration pipeline from entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Data Loading and Profiling](#part-1-data-loading-and-profiling)
- [Part 2: Entity Matching](#part-2-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
  - [Step 5: Machine Learning-based Matching Rules](#step-5-machine-learning-based-matching-rules)
- [Part 3: Data Fusion](#part-3-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

### Datasets

- **DBpedia**: 10,086 records
- **Forbes**: 2,000 records
- **Fullcontact**: 1,931 records

In [1]:
from utils import get_repo_root

ROOT = get_repo_root()
INPUT_DIR = ROOT / "usecases" / "input" / "companies"
OUTPUT_DIR = ROOT / "usecases" / "output" / "companies"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Part 1: Data Loading and Profiling

In [2]:
from PyDI.io import load_xml

# Load DBpedia dataset
dbpedia = load_xml(
    INPUT_DIR / "data" / "dbpedia.xml",
    name="dbpedia",
    nested_handling="aggregate"
)

# Load Forbes dataset
forbes = load_xml(
    INPUT_DIR / "data" / "forbes.xml",
    name="forbes",
    nested_handling="aggregate"
)

# Load Last.fm dataset
fullcontact = load_xml(
    INPUT_DIR / "data" / "fullcontact.xml",
    name="fullcontact",
    nested_handling="aggregate"
)

# Display basic information
datasets = [dbpedia, forbes, fullcontact]
names = ["DBpedia", "Forbes", "FullContact"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 14,017


In [3]:
from PyDI.profiling import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

dbpedia:
  Rows: 10,086
  Columns: 9
  Total nulls: 30,516
  Null percentage: 33.6%
  Null counts per column:
    city: 700 (6.9%)
    industry: 3,208 (31.8%)
    keypeople_name: 9,158 (90.8%)
    assets: 9,426 (93.5%)
    revenue: 8,024 (79.6%)

forbes:
  Rows: 2,000
  Columns: 7
  Total nulls: 114
  Null percentage: 0.8%
  Null counts per column:
    country: 71 (3.5%)
    industry: 43 (2.1%)

fullcontact:
  Rows: 1,931
  Columns: 6
  Total nulls: 3,586
  Null percentage: 31.0%
  Null counts per column:
    country: 508 (26.3%)
    city: 464 (24.0%)
    keypeople_name: 1,739 (90.1%)
    founded: 875 (45.3%)



{'rows': 1931,
 'columns': 6,
 'nulls_total': 3586,
 'nulls_per_column': {'id': 0,
  'name': 0,
  'country': 508,
  'city': 464,
  'keypeople_name': 1739,
  'founded': 875},
 'dtypes': {'id': 'object',
  'name': 'object',
  'country': 'object',
  'city': 'object',
  'keypeople_name': 'object',
  'founded': 'object'}}

### Attribute Coverage Analysis

In [4]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

📊 Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,forbes_count,forbes_pct,forbes_coverage,forbes_samples,fullcontact_count,fullcontact_pct,fullcontact_coverage,fullcontact_samples,avg_coverage,max_coverage,datasets_with_attribute
0,assets,660/10086,6.5%,0.065437,"['8', '240560000000', '607100000']",2000/2000,100.0%,1.0000,"['3124900000000', '2449500000000', '2405400000...",0/0,0%,0.000000,N/A,0.355146,1.000000,2
1,city,9386/10086,93.1%,0.930597,"['Castelnaudary', 'Lisbon', 'Mexico City']",0/0,0%,0.0000,N/A,1467/1931,76.0%,0.759710,"['Brooklyn', 'Toronto', 'Waterloo']",0.563436,0.930597,2
2,country,10086/10086,100.0%,1.000000,"['France', 'Portugal', 'Mexico']",1929/2000,96.5%,0.9645,"['China', 'China', 'China']",1423/1931,73.7%,0.736924,"['United States', 'Canada', 'United States']",0.900475,1.000000,3
3,founded,10086/10086,100.0%,1.000000,"['1970-01-01', '1993-01-01', '2002-01-01']",0/0,0%,0.0000,N/A,1056/1931,54.7%,0.546867,"['1908-01-01', '1957-01-01', '1871-01-01']",0.515622,1.000000,2
4,id,10086/10086,100.0%,1.000000,['http://dbpedia.org/resource/%C3%80_la_Table_...,2000/2000,100.0%,1.0000,"['http://www.forbes.com/companies/icbc/', 'htt...",1931/1931,100.0%,1.000000,"['fullcontact_1', 'fullcontact_2', 'fullcontac...",1.000000,1.000000,3
5,industry,6878/10086,68.2%,0.681935,"['Meat', 'Animation', 'Communication']",1957/2000,97.9%,0.9785,"['Major Banks', 'Regional Banks', 'Regional Ba...",0/0,0%,0.000000,N/A,0.553478,0.978500,2
6,keypeople_name,928/10086,9.2%,0.092009,"['Çalık Holding', 'Ahmet Çalık', 'Marcel Paul']",0/0,0%,0.0000,N/A,192/1931,9.9%,0.099430,"['Raphael Bemporad', 'John Pitcairn', 'Douglas...",0.063813,0.099430,2
7,name,10086/10086,100.0%,1.000000,"['À la Table de Spanghero', '�?guas de Portuga...",2000/2000,100.0%,1.0000,"['ICBC', 'China Construction Bank', 'Agricultu...",1931/1931,100.0%,1.000000,"['BBMG', 'CIT Group Inc (DEL)', 'City & Nation...",1.000000,1.000000,3
8,revenue,2062/10086,20.4%,0.204442,"['2.8', '65170000000', '358500000']",2000/2000,100.0%,1.0000,"['148700000000', '121300000000', '136400000000']",0/0,0%,0.000000,N/A,0.401481,1.000000,2
9,website,0/0,0%,0.000000,N/A,2000/2000,100.0%,1.0000,"['http://www.forbes.com/companies/icbc/', 'htt...",0/0,0%,0.000000,N/A,0.333333,1.000000,1



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['assets', 'city', 'country', 'founded', 'id', 'industry', 'keypeople_name', 'name', 'revenue']


### Detailed Data Profiling

In [6]:
from pathlib import Path

# Generate detailed HTML profiles for each dataset
profile_dir = OUTPUT_DIR / "dataset-profiles"
profile_dir.mkdir(parents=True, exist_ok=True)

profile_paths = []

for df, name in zip(datasets, names):
    print(f"Profiling {name}...")
    
    profile_path = profiler.profile(df, str(profile_dir))
    profile_paths.append(profile_path)
    print(f"Profile saved: {profile_path}")

print(f"\n Generated {len(profile_paths)} detailed HTML reports")
print(f" Location: {profile_dir}")
print("\n Open these HTML files in your browser for interactive exploration:")
for path in profile_paths:
    print(f"  • {Path(path).name}")


Profiling DBpedia...


/Users/luca/PycharmProjects/PyDI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO ] visions.backends - Pandas backend loaded 2.3.2
[INFO ] visions.backends - Numpy backend loaded 1.26.4
[INFO ] visions.backends - Pyspark backend NOT loaded
[INFO ] visions.backends - Python backend loaded


Summarize dataset:  60%|██████    | 12/20 [00:01<00:00,  9.72it/s, scatter assets, assets]    [INFO ] matplotlib.category - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[INFO ] matplotlib.category - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[INFO ] matplotlib.category - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[INFO ] matplotlib.category - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
Summarize dataset:  65%|██

Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/dataset-profiles/dbpedia_profile.html
Profiling Forbes...


Summarize dataset:  56%|█████▌    | 10/18 [00:00<00:00, 17.32it/s, scatter assets, assets]    [INFO ] matplotlib.category - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[INFO ] matplotlib.category - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
Summarize dataset:  61%|██████    | 11/18 [00:06<00:00, 17.32it/s, scatter revenue, assets][INFO ] matplotlib.category - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[INFO ] matplotlib.category - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotte

Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/dataset-profiles/forbes_profile.html
Profiling FullContact...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 15.72it/s]

Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/dataset-profiles/fullcontact_profile.html

 Generated 3 detailed HTML reports
 Location: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/dataset-profiles

 Open these HTML files in your browser for interactive exploration:
  • dbpedia_profile.html
  • forbes_profile.html
  • fullcontact_profile.html


## Part 2: Entity Matching

### Step 1: Blocking

In [5]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [26]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

# Standard Blocking - First meaningful token in the name
def generate_blocking_keys_tokens(company_name: str):
    tokens = re.split(r'[^a-z]', company_name.lower())
    first_token = [token for token in tokens if len(token) > 1]
    if first_token:
        return first_token[0]
    else:
        return company_name # Return full string if no valid token found

# Add first-token column to the original dataframes used for blocking
dbpedia['name_first_token'] = dbpedia['name'].apply(generate_blocking_keys_tokens)
forbes['name_first_token'] = forbes['name'].apply(generate_blocking_keys_tokens)
fullcontact['name_first_token'] = fullcontact['name'].apply(generate_blocking_keys_tokens)

standard_blocker_f2d = StandardBlocker(
    forbes, dbpedia,
    on=['name_first_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
standard_candidates_f2d = standard_blocker_f2d.materialize()

sn_blocker_f2d = SortedNeighbourhoodBlocker(
    forbes, dbpedia,
    key='name',  # Sort by name
    window=20,     # Compare with 20 neighbors
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
sn_candidates_f2d = sn_blocker_f2d.materialize()

token_blocker_f2d = TokenBlocker(
    forbes, dbpedia,
    column='name',      # Tokenize names
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id',
    ngram_size=3,
    ngram_type='character'
)
token_candidates_f2d = token_blocker_f2d.materialize()

embedding_blocker_f2d = EmbeddingBlocker(
    forbes, dbpedia,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,          # Top 20 most similar
    batch_size=500,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_f2d = embedding_blocker_f2d.materialize()

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1647 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 7922 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 658 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created sorted neighbourhood with window size 20
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created 1 sorted sequence from 12086 records
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation

### Step 2: Evaluate Blocking Against Ground Truth

In [ ]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_f2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root -   Pair Completeness: 0.952
[INFO ] root -   Pair Quality:      0.013
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 99/104
[INFO ] root -   Batches Processed:  8
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9519230769230769,
 'pair_quality': 0.01326544285140024,
 'reduction_ratio': 0.9996300317271466,
 'total_candidates': 7463,
 'total_possible_pairs': 20172000,
 'true_positives_found': 99,
 'total_true_pairs': 104,
 'batches_processed': 8,
 'evaluation_timestamp': '2025-10-28T16:18:07.656043',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_detailed_results.csv']}

In [27]:
standard_blocker_f2fc = StandardBlocker(
    forbes, fullcontact,
    on=['name_first_token'],  # Block on first token in name
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
standard_candidates_f2fc = standard_blocker_f2fc.materialize()

sn_blocker_f2fc = SortedNeighbourhoodBlocker(
    forbes, fullcontact,
    key='name',  # Sort by name
    window=20,     # Compare with 20 neighbors
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
sn_candidates_f2fc = sn_blocker_f2fc.materialize()

token_blocker_f2fc = TokenBlocker(
    forbes, fullcontact,
    column='name',      # Tokenize names
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id',
    ngram_size=3,
    ngram_type='character'
)
token_candidates_f2fc = token_blocker_f2fc.materialize()

embedding_blocker_f2fc = EmbeddingBlocker(
    forbes, fullcontact,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,          # Top 20 most similar
    batch_size=500,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_f2fc = embedding_blocker_f2fc.materialize()

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1647 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1750 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 776 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created sorted neighbourhood with window size 20
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created 1 sorted sequence from 3931 records
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/

Now let's evaluate which blocking method we want to use for each dataset combination:

In [ ]:
# Evaluate all blocking methods for both dataset combinations
evaluator = EntityMatchingEvaluator()

# Create dictionaries of candidates for both dataset combinations
f2d_blocking_candidates = {
    'StandardBlocking': [standard_candidates_f2d, standard_blocker_f2d],
    'SortedNeighbourhoodBlocker': [sn_candidates_f2d, sn_blocker_f2d],
    'TokenBlocking': [token_candidates_f2d, token_blocker_f2d],
    'EmbeddingBlocking': [embedding_candidates_f2d, embedding_blocker_f2d]
}

# Load correspondences for evaluation
f2d_correspondences = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_test.csv",
    name="f2d_test", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Evaluate blocking for f2d datasets
f2d_results = []
for method_name, candidates in f2d_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], f2d_correspondences, candidates[1], out_dir=OUTPUT_DIR / "blocking-evaluation")
    result['method'] = method_name
    result['dataset'] = 'f2d'
    f2d_results.append(result)

# Select best method for each dataset (highest pair_completeness, then highest reduction_ratio)
f2d_best = max(f2d_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))

print(f"Best blocking for f2d: {f2d_best['method']} (PC: {f2d_best['pair_completeness']:.3f}, RR: {f2d_best['reduction_ratio']:.3f})")

[INFO ] root -   Pair Completeness: 0.928
[INFO ] root -   Pair Quality:      0.009
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 64/69
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.942
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   0.997
[INFO ] root -   True Matches Found: 65/69
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.986
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   0.919
[INFO ] root -   True Matches Found: 68/69
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.971
[INFO ] root -   Pair Quality:      0.002
[INFO ] root -   Reduction Ratio:   0.998
[INFO ] root -   True Matches Found: 67/69
[INFO ] root - Blocking evaluation complete!


Best blocking for f2d: TokenBlocking (PC: 0.986, RR: 0.919)


In [ ]:
f2fc_blocking_candidates = {
    'StandardBlocking': [standard_candidates_f2fc, standard_blocker_f2fc],
    'SortedNeighbourhood': [sn_candidates_f2fc, sn_blocker_f2fc],
    'TokenBlocking': [token_candidates_f2fc, token_blocker_f2fc],
    'EmbeddingBlocking': [embedding_candidates_f2fc, embedding_blocker_f2fc]
}

f2fc_correspondences = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_test.csv",
    name="f2fc_test", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Evaluate blocking for f2fc datasets
f2fc_results = []
for method_name, candidates in f2fc_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], f2fc_correspondences, candidates[1], out_dir=OUTPUT_DIR / "blocking-evaluation")
    result['method'] = method_name
    result['dataset'] = 'm2l'
    f2fc_results.append(result)

f2fc_best = max(f2fc_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))

print(f"Best blocking for f2fc: {f2fc_best['method']} (PC: {f2fc_best['pair_completeness']:.3f}, RR: {f2fc_best['reduction_ratio']:.3f})")

[INFO ] root -   Pair Completeness: 0.841
[INFO ] root -   Pair Quality:      0.078
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 122/145
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.883
[INFO ] root -   Pair Quality:      0.003
[INFO ] root -   Reduction Ratio:   0.990
[INFO ] root -   True Matches Found: 128/145
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.979
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   0.921
[INFO ] root -   True Matches Found: 142/145
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.966
[INFO ] root -   Pair Quality:      0.004
[INFO ] root -   Reduction Ratio:   0.990
[INFO ] root -   True Matches Found: 140/145
[INFO ] root - Blocking evaluation complete!


Best blocking for f2fc: TokenBlocking (PC: 0.979, RR: 0.921)


### Step 3: Entity Matching with Comparators

In [33]:
from PyDI.entitymatching import StringComparator

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='industry',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_f2d = matcher.match(
    df_left=forbes,
    df_right=dbpedia, 
    candidates=standard_blocker_f2d, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators,
    weights=[1.0, 0.5, 1.0],
    threshold=0.1,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 2000 x 10086 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 2000 x 10086 elements after 0:00:0.027; 7463 blocked pairs (reduction ratio: 0.9996300317271466)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:2.394; found 7375 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [ ]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  64
[INFO ] root -   True Negatives:  46
[INFO ] root -   False Positives: 25
[INFO ] root -   False Negatives: 5
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.786
[INFO ] root -   Precision: 0.719
[INFO ] root -   Recall:    0.928
[INFO ] root -   F1-Score:  0.810


{'precision': 0.7191011235955056,
 'recall': 0.927536231884058,
 'f1': 0.8101265822784809,
 'accuracy': 0.7857142857142857,
 'true_positives': 64,
 'false_positives': 25,
 'false_negatives': 5,
 'true_negatives': 46,
 'threshold_used': 0.0,
 'total_correspondences': 7375,
 'filtered_correspondences': 7375,
 'evaluation_timestamp': '2025-10-28T16:35:51.803501',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_detailed_results.csv']}

In [36]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

Analyzing cluster size distribution in our entity matching results...


[INFO ] root - Cluster Size Distribution of 645 clusters:
[INFO ] root - 	Cluster Size	| Frequency	| Percentage
[INFO ] root - 	──────────────────────────────────────────────────
[INFO ] root - 		2	|	368	|	57.05%
[INFO ] root - 		3	|	115	|	17.83%
[INFO ] root - 		4	|	52	|	8.06%
[INFO ] root - 		5	|	26	|	4.03%
[INFO ] root - 		6	|	14	|	2.17%
[INFO ] root - 		7	|	17	|	2.64%
[INFO ] root - 		8	|	6	|	0.93%
[INFO ] root - 		9	|	9	|	1.40%
[INFO ] root - 		10	|	9	|	1.40%
[INFO ] root - 		11	|	1	|	0.16%
[INFO ] root - 		12	|	6	|	0.93%
[INFO ] root - 		13	|	1	|	0.16%
[INFO ] root - 		14	|	3	|	0.47%
[INFO ] root - 		15	|	5	|	0.78%
[INFO ] root - 		16	|	1	|	0.16%
[INFO ] root - 		20	|	1	|	0.16%
[INFO ] root - 		21	|	1	|	0.16%
[INFO ] root - 		22	|	1	|	0.16%
[INFO ] root - 		23	|	1	|	0.16%
[INFO ] root - 		25	|	2	|	0.31%
[INFO ] root - 		26	|	1	|	0.16%
[INFO ] root - 		32	|	1	|	0.16%
[INFO ] root - 		34	|	1	|	0.16%
[INFO ] root - 		49	|	1	|	0.16%
[INFO ] root - 		56	|	1	|	0.16%
[INFO ] root - 		11


📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,368,57.054264
1,3,115,17.829457
2,4,52,8.062016
3,5,26,4.031008
4,6,14,2.170543
5,7,17,2.635659
6,8,6,0.930233
7,9,9,1.395349
8,10,9,1.395349
9,11,1,0.155039


In [37]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_f2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 645 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [83]:
from PyDI.entitymatching import MaximumBipartiteMatching

# use Maximum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
correspondences_f2d = clusterer.cluster(correspondences_f2d)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] root - Filtered correspondences: 886 -> 886 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 886 -> 886 
[INFO ] root - MaximumBipartiteMatching: 886 -> 886 correspondences
[INFO ] root - MaximumBipartiteMatching: 1772 -> 1772 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  47
[INFO ] root -   True Negatives:  69
[INFO ] root -   False Positives: 2
[INFO ] root -   False Negatives: 22
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.829
[INFO ] root -   Precision: 0.959
[INFO ] root -   Recall:    0.681
[INFO ] root -   F1-Score:  0.797


{'precision': 0.9591836734693877,
 'recall': 0.6811594202898551,
 'f1': 0.7966101694915255,
 'accuracy': 0.8285714285714286,
 'true_positives': 47,
 'false_positives': 2,
 'false_negatives': 22,
 'true_negatives': 69,
 'threshold_used': 0.0,
 'total_correspondences': 886,
 'filtered_correspondences': 886,
 'evaluation_timestamp': '2025-10-28T17:49:05.249533',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_detailed_results.csv']}

[INFO ] root - Cluster Size Distribution of 886 clusters:
[INFO ] root - 	Cluster Size	| Frequency	| Percentage
[INFO ] root - 	──────────────────────────────────────────────────
[INFO ] root - 		2	|	886	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/cluster_analysis/cluster_size_distribution.csv



📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,886,100.0


### Step 5: Machine Learning-based Matching Rules

In [ ]:
from PyDI.entitymatching import FeatureExtractor
from PyDI.entitymatching.comparators import DateComparator

# Load ground truth correspondences
f2fc_train = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_train.csv",
    name="ground_truth_train",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

f2fc_test = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_test.csv",
    name="ground_truth_test",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

similarity_comparators = [
    # Name similarity features - most important for movie matching
    StringComparator("name", similarity_function="jaccard", preprocess=normalize_text),
    StringComparator("name", similarity_function="levenshtein", preprocess=normalize_text),
    StringComparator("name", similarity_function="cosine", preprocess=normalize_text),

    # Country similarity
    StringComparator("country", similarity_function="jaccard", preprocess=normalize_text),
    StringComparator("country", similarity_function="levenshtein", preprocess=normalize_text),
]

feature_extractor = FeatureExtractor(similarity_comparators)

# Extract features using FeatureExtractor
train_features = feature_extractor.create_features(
    forbes, fullcontact, f2fc_train[['id1', 'id2']], labels=f2fc_train['label'], id_column='id'
)

print(f"✅ Training features extracted!")
print(f"Feature columns: {[col for col in train_features.columns if col not in ['id1', 'id2', 'label']]}")

# Prepare data for ML training
feature_columns = [col for col in train_features.columns if col not in ['id1', 'id2', 'label']]

X_train = train_features[feature_columns]
y_train = train_features['label']

print(f"Training data: X={X_train.shape}, y={y_train.shape}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")

[INFO ] root - Label distribution: 475 positive, 1038 negative


✅ Training features extracted!
Feature columns: ['StringComparator(name, jaccard, tokenization=word, list_strategy=None)', 'StringComparator(name, levenshtein, tokenization=char, list_strategy=None)', 'StringComparator(name, cosine, tokenization=word, list_strategy=None)', 'StringComparator(country, jaccard, tokenization=word, list_strategy=None)', 'StringComparator(country, levenshtein, tokenization=char, list_strategy=None)']
Training data: X=(1513, 5), y=(1513,)
Class distribution: {False: 1038, True: 475}


#### Full Scikit-learn integration

In [41]:
# Set up GridSearchCV with multiple models and hyperparameters
print(f"\n🔍 Setting up GridSearchCV...")

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

# Define models and parameter grids
param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None]
        }
    }
}

# Use F1 score as the scoring metric (good for imbalanced data)
scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")

# Train models using GridSearchCV
print(f"\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    

    # Create GridSearchCV
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,  # Use all available cores
        verbose=0
    )
    
    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)
    
    # Store results
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    # Track overall best model
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")


🔍 Setting up GridSearchCV...
GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

  ✅ RandomForest: Best CV F1 = 0.9015
     Best params: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.8523
     Best params: {'C': 10.0, 'class_weight': None, 'penalty': 'l2'}

Training GradientBoosting...
  ✅ GradientBoosting: Best CV F1 = 0.9009
     Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}

Training SVM...
  ✅ SVM: Best CV F1 = 0.8793
     Best params: {'C': 10.0, 'class_weight': None, 'kernel': 'rbf'}

🏆 Best Overall Model: RandomForest (CV F1: 0.9015)


Now, we can directly use the trained model with PyDIs MLBasedMatcher

In [86]:
from PyDI.entitymatching import MLBasedMatcher

# Create MLBasedMatcher and apply trained model
ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_f2fc = ml_matcher.match(
    forbes, fullcontact, candidates=standard_blocker_f2fc, id_column='id', trained_classifier=best_overall_model
)

[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 2000 x 1931 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 2000 x 1931 elements after 0:00:0.004; 1563 blocked pairs (reduction ratio: 0.9995952874158467)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:0.787; found 1148 correspondences.


In [87]:
# Show feature importance if available
if hasattr(best_overall_model, 'feature_importances_'):
    print(f"\n🔍 Top Feature Importances:")
    importance_df = ml_matcher.get_feature_importance(best_overall_model, feature_columns)
    display(importance_df.head(8))


🔍 Top Feature Importances:


,feature,importance
2,"StringComparator(name, cosine, tokenization=wo...",0.4228
1,"StringComparator(name, levenshtein, tokenizati...",0.2709
0,"StringComparator(name, jaccard, tokenization=w...",0.2423
4,"StringComparator(country, levenshtein, tokeniz...",0.0406
3,"StringComparator(country, jaccard, tokenizatio...",0.0235


In [88]:
eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=f2fc_test,
    out_dir=debug_output_dir
)

display(eval_results)

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
    out_dir=OUTPUT_DIR / "cluster_analysis"
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  113
[INFO ] root -   True Negatives:  309
[INFO ] root -   False Positives: 5
[INFO ] root -   False Negatives: 32
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.919
[INFO ] root -   Precision: 0.958
[INFO ] root -   Recall:    0.779
[INFO ] root -   F1-Score:  0.859


{'precision': 0.9576271186440678,
 'recall': 0.7793103448275862,
 'f1': 0.8593155893536121,
 'accuracy': 0.9193899782135077,
 'true_positives': 113,
 'false_positives': 5,
 'false_negatives': 32,
 'true_negatives': 309,
 'threshold_used': 0.0,
 'total_correspondences': 1148,
 'filtered_correspondences': 1148,
 'evaluation_timestamp': '2025-10-28T17:52:00.402651',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_detailed_results.csv']}

[INFO ] root - Cluster Size Distribution of 733 clusters:
[INFO ] root - 	Cluster Size	| Frequency	| Percentage
[INFO ] root - 	──────────────────────────────────────────────────
[INFO ] root - 		2	|	655	|	89.36%
[INFO ] root - 		3	|	44	|	6.00%
[INFO ] root - 		4	|	8	|	1.09%
[INFO ] root - 		5	|	7	|	0.95%
[INFO ] root - 		6	|	4	|	0.55%
[INFO ] root - 		7	|	2	|	0.27%
[INFO ] root - 		8	|	3	|	0.41%
[INFO ] root - 		9	|	2	|	0.27%
[INFO ] root - 		10	|	1	|	0.14%
[INFO ] root - 		11	|	2	|	0.27%
[INFO ] root - 		12	|	1	|	0.14%
[INFO ] root - 		13	|	1	|	0.14%
[INFO ] root - 		15	|	2	|	0.27%
[INFO ] root - 		38	|	1	|	0.14%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/cluster_analysis/cluster_size_distribution.csv



📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,655,89.358799
1,3,44,6.002729
2,4,8,1.091405
3,5,7,0.954980
4,6,4,0.545703
5,7,2,0.272851
6,8,3,0.409277
7,9,2,0.272851
8,10,1,0.136426
9,11,2,0.272851


## Part 3: Data Fusion

In [94]:
forbes["forbes_id"] = forbes["id"]

# Assign trust scores to datasets
forbes.attrs["trust_score"] = 3
dbpedia.attrs["trust_score"] = 1
fullcontact.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_f2d, correspondences_f2fc], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 2,034


## Step 1: Define Fusion Strategy

In [99]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, union, prefer_higher_trust, voting, maximum, most_recent

strategy = DataFusionStrategy('company_fusion_strategy')

strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('assets', prefer_higher_trust)
strategy.add_attribute_fuser('revenue', prefer_higher_trust)
strategy.add_attribute_fuser('keypeople_name', union)
strategy.add_attribute_fuser('founded', voting)
strategy.add_attribute_fuser('country', voting)
strategy.add_attribute_fuser('city', shortest_string)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'assets' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'revenue' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'keypeople_name' using rule 'union'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'founded' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'country' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'city' using rule 'shortest_string'


## Step 2: Run Fusion

In [100]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[forbes, dbpedia, fullcontact],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'company_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 3067 of 3067 unique IDs
[INFO ] PyDI.fusion.engine - Created 12149 record groups from 2034 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 12149 groups:
[INFO ] PyDI.fusion.engine -     Group Size | Frequency
[INFO ] PyDI.fusion.engine -     ----------------------
[INFO ] PyDI.fusion.engine -           1 |   10949
[INFO ] PyDI.fusion.engine -           2 |     876
[INFO ] PyDI.fusion.engine -           3 |     260
[INFO ] PyDI.fusion.engine -           4 |      22
[INFO ] PyDI.fusion.engine -           5 |      14
[INFO ] PyDI.fusion.engine -           6 |       3
[INFO ] 

Fused rows: 1,200


,_id,_fusion_group_id,_fusion_sources,forbes_id,industry,city,name,website,assets,name_first_token,country,founded,keypeople_name,id,revenue,_fusion_confidence,_fusion_metadata
0,fullcontact_346,group_0,"[forbes, fullcontact]",http://www.forbes.com/companies/gemalto/,Electronics,Amsterdam,Gemalto,http://www.forbes.com/companies/gemalto/,4000000000,gemalto,Netherlands,2006-01-01,None,fullcontact_346,3200000000,0.727273,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
1,http://www.forbes.com/companies/hugo-boss/,group_1,"[forbes, fullcontact]",http://www.forbes.com/companies/hugo-boss/,Apparel/Accessories,Metzingen,Hugo Boss,http://www.forbes.com/companies/hugo-boss/,2100000000,hugo,Germany,None,None,http://www.forbes.com/companies/hugo-boss/,3200000000,0.590909,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
2,http://www.forbes.com/companies/pentair/,group_2,"[forbes, fullcontact]",http://www.forbes.com/companies/pentair/,Other Industrial Equipment,Manchester,Pentair,http://www.forbes.com/companies/pentair/,11700000000,pentair,Switzerland,1966-01-01,None,http://www.forbes.com/companies/pentair/,7500000000,0.681818,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
3,http://www.forbes.com/companies/pfizer/,group_3,"[forbes, fullcontact]",http://www.forbes.com/companies/pfizer/,Pharmaceuticals,New York,Pfizer,http://www.forbes.com/companies/pfizer/,172100000000,pfizer,United States of America,1848-01-01,None,http://www.forbes.com/companies/pfizer/,52700000000,0.681818,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
4,http://www.forbes.com/companies/hutchison-wham...,group_4,"[forbes, dbpedia]",http://www.forbes.com/companies/hutchison-wham...,Conglomerates,Hong Kong,Hutchison Whampoa,http://www.forbes.com/companies/hutchison-wham...,105200000000,hutchison,Hong Kong,2001-01-01,None,http://www.forbes.com/companies/hutchison-wham...,33000000000,0.636364,"{'_id_rule': 'first_non_null', '_id_inputs': [..."


## Step 3: Evaluate Data Fusion

In [101]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("assets", tokenized_match)
strategy.add_evaluation_function("revenue", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("assets", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("keypeople_name", set_equality_match)
strategy.add_evaluation_function("founded", year_only_match)
strategy.add_evaluation_function("country", tokenized_match)
strategy.add_evaluation_function("city", tokenized_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'revenue' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'keypeople_name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'founded'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'country'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'city'


In [102]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='forbes_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[WARNING] PyDI.fusion.evaluation - Missing 1 expected/reference records in fused dataset: http://www.forbes.com/companies/china-shenhua-energy/
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.650 overall accuracy (76/117)


Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.650
  macro_accuracy: 0.619
  num_evaluated_records: 18
  num_evaluated_attributes: 7
  total_evaluations: 117
  total_correct: 76
  city_accuracy: 0.722
  city_count: 18
  assets_accuracy: 0.611
  assets_count: 18
  founded_accuracy: 0.556
  founded_count: 18
  country_accuracy: 0.889
  country_count: 18
  keypeople_name_accuracy: 0.222
  keypeople_name_count: 9
  name_accuracy: 0.778
  name_count: 18
  revenue_accuracy: 0.556
  revenue_count: 18

Overall Accuracy: 65.0%
